In [18]:
import io
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display


In [19]:
PROJECT_DIR = Path.cwd()
CHECKPOINT_PATH = PROJECT_DIR / "resultsV2" / "checkpoints" / "generator_final.pth"
OUTPUT_DIR = PROJECT_DIR / "resultsV2" / "gen_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENT_DIM = 100

TYPE_LABELS = [
    "Bug",
    "Dark",
    "Dragon",
    "Electric",
    "Fairy",
    "Fighting",
    "Fire",
    "Flying",
    "Ghost",
    "Grass",
    "Ground",
    "Ice",
    "Normal",
    "Poison",
    "Psychic",
    "Rock",
    "Steel",
    "Water",
]

print(f"Proyecto: {PROJECT_DIR}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Dispositivo: {DEVICE}")


Proyecto: c:\Users\danyq\Documents\Semestre8\PokemonGAN
Checkpoint: c:\Users\danyq\Documents\Semestre8\PokemonGAN\resultsV2\checkpoints\generator_final.pth
Dispositivo: cpu


In [20]:
class Generator(nn.Module):
    """Generador compatible con el checkpoint V2."""

    def __init__(self, dim_z, n_labels):
        super().__init__()
        self.dim_z = dim_z
        self.n_labels = n_labels
        self.latent_dim = dim_z + n_labels

        self.linear = nn.Sequential(
            nn.LazyConvTranspose2d(1024, kernel_size=6, stride=1, padding=0),
            nn.LeakyReLU(0.2),
        )

        self.backbone = nn.Sequential(
            self.upsample_block(512),
            self.upsample_block(256),
            self.upsample_block(128),
        )

        self.gen = nn.Sequential(
            nn.LazyConvTranspose2d(3, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def upsample_block(self, num_filters):
        return nn.Sequential(
            nn.LazyConvTranspose2d(num_filters, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(num_filters),
            nn.LeakyReLU(0.2),
            nn.LazyConv2d(num_filters, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(num_filters),
            nn.LeakyReLU(0.2),
            nn.LazyConv2d(num_filters, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(num_filters),
            nn.LeakyReLU(0.2),
        )

    def forward(self, noise, labels):
        noise_with_labels = torch.concat((noise, labels[:, :, None, None]), dim=1)
        init_feat_map = self.linear(noise_with_labels)
        final_feat_map = self.backbone(init_feat_map)
        return self.gen(final_feat_map)


def build_generator():
    generator = Generator(dim_z=LATENT_DIM, n_labels=len(TYPE_LABELS) + 1).to(DEVICE)
    with torch.no_grad():
        warmup_noise = torch.randn(1, LATENT_DIM, 1, 1, device=DEVICE)
        warmup_labels = torch.zeros(1, len(TYPE_LABELS) + 1, device=DEVICE)
        warmup_labels[:, 0] = 1.0
        generator(warmup_noise, warmup_labels)
    return generator


generator = build_generator()


In [21]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No se encontró el checkpoint en: {CHECKPOINT_PATH}")

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = {key.replace("module.", ""): value for key, value in state_dict.items()}
generator.load_state_dict(state_dict, strict=True)
generator.eval()

print("Generador cargado correctamente.")
print(f"Tipos disponibles: {', '.join(TYPE_LABELS)}")


Generador cargado correctamente.
Tipos disponibles: Bug, Dark, Dragon, Electric, Fairy, Fighting, Fire, Flying, Ghost, Grass, Ground, Ice, Normal, Poison, Psychic, Rock, Steel, Water


In [ ]:
def build_condition_vector(primary_type, secondary_type=None, orientation="Front", batch_size=64):
    labels = torch.zeros(batch_size, len(TYPE_LABELS) + 1, device=DEVICE)
    labels[:, 0] = 1.0 if orientation == "Front" else 0.0
    labels[:, 1 + TYPE_LABELS.index(primary_type)] = 1.0
    if secondary_type is not None and secondary_type != "None":
        labels[:, 1 + TYPE_LABELS.index(secondary_type)] = 0.7
    return labels


def generate_grid(primary_type, secondary_type=None, orientation="Front", grid_size=4, seed=None, save=True):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if DEVICE.type == "cuda":
            torch.cuda.manual_seed_all(seed)

    batch_size = grid_size * grid_size
    noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
    labels = build_condition_vector(primary_type, secondary_type=secondary_type, orientation=orientation, batch_size=batch_size)

    with torch.no_grad():
        gen_images = generator(noise, labels).detach().cpu()

    gen_images = (gen_images * 0.5 + 0.5).clamp(0, 1)
    grid = torchvision.utils.make_grid(gen_images, nrow=grid_size)

    title = primary_type if secondary_type in (None, "None") else f"{primary_type}+{secondary_type}"
    file_name = f"{orientation.lower()}_{title}_seed{seed if seed is not None else 'random'}.png".replace(" ", "_")
    file_path = OUTPUT_DIR / file_name

    fig, ax = plt.subplots(figsize=(grid_size * 2, grid_size * 2))
    ax.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    ax.set_title(f"Pokémon generados: {title} | {orientation}", fontsize=14)
    ax.axis("off")

    if save:
        fig.savefig(file_path, bbox_inches="tight", dpi=200)
    plt.show()
    plt.close(fig)

    return gen_images, file_path


primary_dropdown = widgets.Dropdown(options=TYPE_LABELS, value="Fire", description="Tipo 1:")
secondary_dropdown = widgets.Dropdown(options=["None"] + TYPE_LABELS, value="None", description="Tipo 2:")
orientation_radio = widgets.ToggleButtons(options=["Front", "Back"], value="Front", description="Orient:")
grid_slider = widgets.IntSlider(value=4, min=1, max=8, step=1, description="Grid:")
seed_text = widgets.IntText(value=42, description="Seed:")
run_button = widgets.Button(description="Generate", button_style="primary")
output = widgets.Output()


def on_run_clicked(_):
    with output:
        output.clear_output(wait=True)
        secondary_type = None if secondary_dropdown.value == "None" else secondary_dropdown.value
        _, file_path = generate_grid(
            primary_type=primary_dropdown.value,
            secondary_type=secondary_type,
            orientation=orientation_radio.value,
            grid_size=grid_slider.value,
            seed=seed_text.value,
            save=True,
        )
        print(f"Imagen guardada en: {file_path}")


run_button.on_click(on_run_clicked)
controls_top = widgets.HBox([primary_dropdown, secondary_dropdown, orientation_radio])
controls_bottom = widgets.HBox([grid_slider, seed_text, run_button])
display(controls_top, controls_bottom, output)
on_run_clicked(None)


Output()